In [43]:
from dataclasses import dataclass
from openai import OpenAI
import os
import json
from typing import cast
from openai.types.chat import ChatCompletionToolParam ,ChatCompletionMessageParam
from pydantic import BaseModel
from typing import Literal


@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests across all inference providers."""

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str

PROVIDERS = [
    
    Provider(
        "Groq",
        "GROQ_API_KEY",
        True,
        "https://api.groq.com/openai/v1",
        "openai/gpt-oss-120b"
    ),
    Provider(
        "OpenRouter",
        "OPENROUTER_API_KEY",
        True,
        "https://openrouter.ai/api/v1",
        "openai/gpt-sol-latest"
    ),
    Provider(
        "OpenAI",
        "OPENAI_API_KEY",
        True,
        None,
        "gpt-4o-mini"
    ),
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(
        f"No provider key set, add one of {expected} "
        "to your environment variables"
    )

def build_client(provider: Provider) -> OpenAI:
    api_key = os.getenv(provider.env_var)

    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url,
    )

def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")

def llm_reply(prompt: str) -> str:
    provider = select_provider()
    print(f"Using {provider.name} provider")
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=50000,
        messages=[{
            "role": "user",
            "content": prompt,
        }]
    )

    return result.choices[0].message.content or ""

Found a provider key.


In [24]:
WEATHER_DB = {
    "london": {"celcius": 22, "sky": "cloudy"},
    "paris": {"celcius": 24, "sky": "sunny"},
    "tallinn": {"celcius": 28, "sky": "sunny"},
    "moscow": {"celcius": 18, "sky": "rainy"},
    "tokyo": {"celcius": 30, "sky": "sunny"},
    "beijing": {"celcius": 26, "sky": "cloudy"},
    "new york": {"celcius": 28, "sky": "sunny"},
    "mumbai": {"celcius": 32, "sky": "sunny"},
    "cape town": {"celcius": 16, "sky": "cloudy"},
    "sydney": {"celcius": 20, "sky": "sunny"},
    "rio de janeiro": {"celcius": 22, "sky": "cloudy"},
    "cairo": {"celcius": 24, "sky": "sunny"},
    "mexico city": {"celcius": 20, "sky": "cloudy"},
    
}

Below is a tool defined manually by us, which can be called on our machine, and llm will just signal should we call this tool or not ?

In [25]:
def lookup_weather(location: str) -> str:
    """Look up the weather for a location."""
    record = WEATHER_DB.get(location.lower())
    if record is None:
        return f"Sorry, I don't know the weather in {location}."
    return f"The weather in {location} is {record['celcius']}C and {record['sky']}."

In [26]:
weather_schema : ChatCompletionToolParam = {
    "type": "function",
    "function": {
        "name": "lookup_weather",
        "description": "Look up the weather for a location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The location to look up the weather for."}
            },
            "required": ["location"]
        }
    }
} # this schema will be given to the llm

TOOLS = {
    "lookup_weather": lookup_weather,
    "stock_price_checker": None
}

In [27]:
def ask_llm_with_tool(prompt: str, *, max_tokens: int = 400):
    """
    Call the LLM with a tool call.
    """
    provider = select_provider()
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
        tools=[weather_schema],
    )
    print(result.choices)
    return result.choices[0].message

In [28]:
msg = ask_llm_with_tool("What is the weather in Tokyo?")

print("final response from llm", msg)

[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_fbfaf20b-1de5-4700-af3b-157e105d0b0d', function=Function(arguments='{"location":"Tokyo"}', name='lookup_weather'), type='function')], reasoning='User asks: "What is the weather in Tokyo?" Need to call lookup_weather with location "Tokyo".'))]
final response from llm ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_fbfaf20b-1de5-4700-af3b-157e105d0b0d', function=Function(arguments='{"location":"Tokyo"}', name='lookup_weather'), type='function')], reasoning='User asks: "What is the weather in Tokyo?" Need to call lookup_weather with location "Tokyo".')


In [35]:
msg = ask_llm_with_tool("What is the weather in london compared to paris?You must call lookup_weather separately for BOTH London and Paris.")

print("final response from llm", msg)

[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_d131267f-b987-4530-ae76-9054dda4b594', function=Function(arguments='{"location":"London"}', name='lookup_weather'), type='function')], reasoning='The user asks: "What is the weather in London compared to Paris? You must call lookup_weather separately for BOTH London and Paris."\n\nWe need to call the function lookup_weather twice: once with location "London" and once with location "Paris". Then compare the weather and provide answer. Let\'s do that.\n\nWe need to call function. Use tool.'))]
final response from llm ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_d131267f-b987-4530-ae76-9054dda4b594', function=Function(argu

In [36]:
if msg.tool_calls:

        print("tool call detected")
        print("Total tool calls", len(msg.tool_calls))
        for tool_call in msg.tool_calls:
            if tool_call.type == "function":
                print("Tool call name", tool_call.function.name)
                print("Tool call arguments", tool_call.function.arguments)
                city = json.loads(tool_call.function.arguments)["location"]
                print(f"Calling tool for {city}")
                tool = TOOLS[tool_call.function.name] # this will give us the function 
                result = tool(city)
                print(result)

                print("-"*100)
else:
    print("No tool call detected")
    

tool call detected
Total tool calls 1
Tool call name lookup_weather
Tool call arguments {"location":"London"}
Calling tool for London
The weather in London is 22C and cloudy.
----------------------------------------------------------------------------------------------------


In [51]:
def ask_llm_with_tool(prompt: str, *, max_tokens: int = 400) ->str:
    """
    Call the LLM with a tool call.
    """
    provider = select_provider()
    client = build_client(provider)
    messages : list[ChatCompletionMessageParam]= [
        {"role": "user", "content": prompt},
    ]

    while True:
        response = client.chat.completions.create(
            model=provider.model,
            max_tokens=max_tokens,
            messages=messages,
            tools=[weather_schema],
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or ""

        messages.append( cast(ChatCompletionMessageParam,{
            "role": "assistant",
            "content": msg.content,
            "tool_calls": msg.tool_calls,
        }))

        # llm is asking us to call a tool
        for tool_call in msg.tool_calls:
            if tool_call.type == "function":
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments) # {"location": "london"}
                tool_call_id = tool_call.id

                if tool_name not in TOOLS:
                    raise ValueError(f"Tool {tool_name} not found")

                tool = TOOLS[tool_name] # actual function
                result = tool(**tool_args) # -> lookup_weather(location="london")

                messages.append(
                    {"role": "tool", "tool_call_id": tool_call_id, "content": result}
                )

        print("All tool calls done, going back to LLM")



In [52]:
ask_llm_with_tool("What is the weather like in London compared to Paris?",max_tokens=1000)

All tool calls done, going back to LLM
All tool calls done, going back to LLM


'London is currently about 22\u202f°C and cloudy, while Paris is a little warmer at roughly 24\u202f°C and enjoying sunny skies. In other words, Paris is both hotter and clearer, whereas London is cooler and overcast.'